In [0]:
dbutils.widgets.text("source_system","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("source_file_format","")
source_system=dbutils.widgets.get("source_system")
table_name=dbutils.widgets.get("table_name")
landing_timestamp=dbutils.widgets.get("landing_timestamp")
source_file_format=dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/adb_medlake/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check whether the landing path (or any parent directory) exists
try:
    dbutils.fs.ls(landing_path)
    path_exists = True
except Exception:
    path_exists = False

# Exit outside the try block so the exit is not swallowed by the except
if not path_exists:
    print(f"No new files are received at {landing_path}")
    dbutils.notebook.exit("No new files are received")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as F
updated_df=landing_df.withColumn("landing_timestamp",F.lit(landing_timestamp))\
.withColumn("insert_timestamp",F.current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_medlake.bronze.{table_name}")